### 🛠️ Environment Setup
In this cell, we install the specialized `arabert` library along with its dependencies like `PyArabic` and `farasapy` to enable advanced preprocessing of Arabic text.

In [ ]:
!pip install arabert

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 15.1 MB/s eta 0:00:00
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186540 sha256=5c29b6d89f4c030e057629cdd065e9446c3ccaa7a06c4bed3d9fc7aa31e69917
  Stored in directory: /root/.cache/pip/wheels/e1/79/2a/9864bac03173079abcc089d3eb3ec87456ac09c381b801f1a3
Successfully built emoji


### 📥 Dataset Loading
Here, we download the multi-lingual Twitter sentiment dataset for Arabic (`mteb/tweet_sentiment_multilingual`) from HuggingFace to extract train and test samples.

In [ ]:
from datasets import load_dataset

# Load an Arabic sentiment dataset from HuggingFace
dataset = load_dataset("mteb/tweet_sentiment_multilingual", "arabic")
#    ↑ This downloads Arabic tweet data with sentiment labels
#    ↑ Each row has: "text" (Arabic text) and "label" (0, 1, 2)

# Look at the data
print(dataset)
print(dataset["train"][0])  # See one example
print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

# Label mapping:
# 0 = negative, 1 = neutral, 2 = positive
label_names = ["negative", "neutral", "positive"]

README.md:   0%|          | 0.00/22.0k [00:00<?, ?B/s]

train/arabic.jsonl.gz: reconstructing file:   0%|          |  0.00B /  116kB            

train/arabic.jsonl.gz: downloading bytes:           |  0.00B            

test/arabic.jsonl.gz: reconstructing file:   0%|          |  0.00B / 49.4kB            

test/arabic.jsonl.gz: downloading bytes:           |  0.00B            

...

validation/arabic.jsonl.gz: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'lang'],
        num_rows: 1839
    })
    test: Dataset({
        features: ['text', 'label', 'lang'],
        num_rows: 870
    })
    validation: Dataset({
        features: ['text', 'label', 'lang'],
        num_rows: 324
    })
})
{'text': 'RT @user: @user @user   وصلنا لاقتصاد اسوء من سوريا والعراق ومن غير حربانجاز ده ولا مش انجاز يا متعلمين يا بتوع المدا…', 'label': '0', 'lang': 'arabic'}
Training samples: 1839
Test samples: 870


### 🔍 Dataset Inspection
We print a sample dictionary from the training set to verify the fields (`text`, `label`, `lang`) and confirm the dataset length.

In [ ]:
print(dataset["train"][2])

{'text': '@user على فكره شركة محترمه حداعطوني كيبل كهديه  ويوم قتله انه شحنه بيكلفني اكثر من قيمته قال دش امازون واختار اي شي قيمته ٣٠ دولار', 'label': '2', 'lang': 'arabic'}


## 🚀 Phase 1: Baseline AraBERT Model Training

In this phase, we set up our tokenizer, preprocess the raw Arabic text using the `ArabertPreprocessor`, and train a baseline `aubmindlab/bert-base-arabertv2` model for Sequence Classification.

### 🤖 Model and Tokenizer Initialization
We load the pre-trained `bert-base-arabertv2` model and its corresponding tokenizer and preprocessor specifically designed to clean and normalize Arabic grammar.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from arabert.preprocess import ArabertPreprocessor

# ================================================================
# WHY AraBERT?
# ================================================================
# Regular BERT was trained on English text. It doesn't understand Arabic.
# AraBERT was trained on 77GB of Arabic text (news, Wikipedia, social media).
# It understands Arabic grammar, dialects, and word meanings.
#
# It's like the difference between hiring someone who speaks Arabic
# vs someone with a phrase book.
# ================================================================

model_name = "aubmindlab/bert-base-arabertv2"
#    ↑ "aubmindlab" = American University of Beirut's AI lab
#    ↑ "bert-base" = the standard size (110M parameters)
#    ↑ "arabertv2" = version 2 (better than v1)

# The preprocessor handles Arabic-specific text cleaning:
# - Removes diacritics (the little marks above Arabic letters)
# - Normalizes different forms of the same letter (أ إ آ → ا)
# - Handles Arabic-specific punctuation
preprocessor = ArabertPreprocessor(model_name)

# The tokenizer converts text → numbers the model can understand
# "المنتج رائع" → [101, 5234, 8721, 102]
tokenizer = AutoTokenizer.from_pretrained(model_name)

# The model itself — a neural network with 110M parameters
# num_labels=3 means: negative, neutral, positive
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
# Should print ~110,000,000 (110 million)

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'farasa-api.qcri.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


100%|██████████| 241M/241M [03:16<00:00, 1.23MiB/s]


[2026-10-09 08:12:25,344 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 135,195,651


### ⚙️ Preprocessing Pipeline
This function applies Arabic text normalization rules, tokenizes strings into numeric representations of length 128, and casts labels to integers.

In [ ]:
def preprocess_function(examples):
    """Convert raw Arabic text into numbers the model understands and cast labels to integers."""
    # Step 1: AraBERT preprocessing (Arabic-specific cleaning)
    texts = [preprocessor.preprocess(text) for text in examples["text"]]

    # Step 2: Tokenize (convert text → number sequences)
    result = tokenizer(
        texts,
        truncation=True,     # Cut text longer than max_length
        max_length=128,      # Maximum 128 tokens (words/subwords)
        padding="max_length", # Pad shorter texts to 128 tokens
    )

    # Step 3: Convert string labels to integers
    if "label" in examples:
        result["label"] = [int(label) for label in examples["label"]]

    return result

# Apply to entire dataset
tokenized = dataset.map(preprocess_function, batched=True)

Parameter 'function'=<function preprocess_function at 0x79d9f04d2200> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/1839 [00:00<?, ? examples/s]

Map:   0%|          | 0/870 [00:00<?, ? examples/s]

Map:   0%|          | 0/324 [00:00<?, ? examples/s]

### 🏋️ Baseline Trainer Setup and Training
We configure training parameters (3 epochs, learning rate of 2e-5, and F1 macro evaluation metric) and initiate training of the baseline model.

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Calculate accuracy and F1 score after each evaluation."""
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    #    ↑ The model outputs 3 numbers (one per class)
    #      argmax picks the highest one as the prediction

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    #    ↑ "macro" = calculate F1 for each class, then average
    #      This is fair even if classes are imbalanced

    return {"accuracy": acc, "f1_macro": f1}

training_args = TrainingArguments(
    output_dir="./",
    num_train_epochs=3,
    #    ↑ 3 passes through the entire dataset
    #      More epochs = better learning but risk overfitting
    per_device_train_batch_size=16,
    #    ↑ Process 16 examples at once
    #      Higher = faster but needs more GPU memory
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    #    ↑ 0.00002 — how big each learning step is
    #      Too high = model oscillates, too low = learns too slowly
    #      2e-5 is the standard for fine-tuning BERT models
    eval_strategy="epoch",
    #    ↑ Evaluate after each epoch (not each step)
    save_strategy="epoch",
    load_best_model_at_end=True,
    #    ↑ After training, automatically load the best checkpoint
    metric_for_best_model="f1_macro",
    #    ↑ "Best" means highest F1 score, not lowest loss
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

# START TRAINING (this takes ~15-30 min on Colab GPU)
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 31}.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.789512,0.673563,0.670032
2,No log,0.732971,0.683908,0.686495
3,No log,0.734897,0.690805,0.692958


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=345, training_loss=0.6981400641842165, metrics={'train_runtime': 177.4454, 'train_samples_per_second': 31.091, 'train_steps_per_second': 1.944, 'total_flos': 362899181401344.0, 'train_loss': 0.6981400641842165, 'epoch': 3.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 💾 Saving Baseline & Writing Reports
We save the baseline tokenizer and model to disk and initialize our first report file (`reports/module-1.md`) with its accuracy and F1 score.

In [ ]:
import os

# Save the fine-tuned model
os.makedirs("models", exist_ok=True)
model.save_pretrained("models/arabert-sentiment")
tokenizer.save_pretrained("models/arabert-sentiment")

# Evaluate on test set
results = trainer.evaluate()
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1 Macro: {results['eval_f1_macro']:.4f}")

# Save metrics for the report
os.makedirs("reports", exist_ok=True)
with open("reports/module-1.md", "w") as f:
    f.write("# Module 1 Report\n\n")
    f.write(f"## Baseline Metrics\n\n")
    f.write(f"- **Accuracy:** {results['eval_accuracy']:.4f}\n")
    f.write(f"- **F1 Macro:** {results['eval_f1_macro']:.4f}\n")
    f.write(f"- **Model:** {model_name}\n")
    f.write(f"- **Epochs:** 3\n")
    f.write(f"- **Learning Rate:** 2e-5\n")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
No log,0.734897,3,0.690805,0.692958


Accuracy: 0.6908
F1 Macro: 0.6930


## 📈 Phase 2: Class Weights, Warmup, and Early Stopping

To improve performance, we implement a sequence of targeted optimization strategies:
1. **Class Weighting:** Dynamically penalize minority class errors if there is an imbalance.
2. **Linear Warmup Steps:** Stabilize training at start.
3. **Early Stopping:** Stop training when evaluation metric stops improving to avoid overfitting.

### ⚖️ Implementing Class Weights and Custom Loss
To prevent majority class bias, we calculate balanced class weights, design a custom trainer using cross-entropy, and apply early stopping callback flags.

In [ ]:
import torch
from torch import nn
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.utils.class_weight import compute_class_weight

# 1. Compute Class Weights to handle label imbalance
train_labels = tokenized["train"]["label"]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=np.array(train_labels)
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computed Class Weights: {class_weights}")

# 2. Define a Custom Trainer to inject class weights into the Loss Function
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 3. Enhanced Training Arguments
improved_training_args = TrainingArguments(
    output_dir="./improved_model",
    num_train_epochs=5,                  # Train longer...
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,                  # Slightly adjusted learning rate
    weight_decay=0.01,                   # Strength of weight decay regularization
    warmup_steps=100,                    # Linear warmup steps instead of ratio
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# Re-initialize the model to reset weights for a clean run
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

# 4. Trainer with Early Stopping
improved_trainer = WeightedTrainer(
    model=model,
    args=improved_training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stop if F1 doesn't improve for 2 epochs
)

# Start improved training
improved_trainer.train()

Computed Class Weights: [1. 1. 1.]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transfor

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.770155,0.682759,0.685633
2,No log,0.737718,0.687356,0.684996
3,No log,0.807659,0.683908,0.685023


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=345, training_loss=0.6852772146031476, metrics={'train_runtime': 207.5503, 'train_samples_per_second': 44.303, 'train_steps_per_second': 2.77, 'total_flos': 362899181401344.0, 'train_loss': 0.6852772146031476, 'epoch': 3.0})

### 📊 Evaluating Class-Weighted Model
We run model validation on our newly fine-tuned class-weighted trainer to evaluate if early stopping and weights improved performance.

In [ ]:
# Evaluate the improved model
improved_results = improved_trainer.evaluate()
print(f"Improved Accuracy: {improved_results['eval_accuracy']:.4f}")
print(f"Improved F1 Macro: {improved_results['eval_f1_macro']:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
No log,0.770155,3,0.682759,0.685633


Improved Accuracy: 0.6828
Improved F1 Macro: 0.6856


## 🏆 Phase 3: Domain-Specific Model Optimization (Twitter-AraBERT) & Cosine Decay

For social media text, utilizing a domain-adapted model trained on tweets yields significant improvements. We switch to `aubmindlab/bert-base-arabertv02-twitter` coupled with a **Cosine Annealing Learning Rate Scheduler** to optimize convergence.

### 🐦 Twitter-Specific Model Pipeline
We load a specialized Twitter-AraBERT model, re-tokenize our tweets, and construct a training routine using a smooth Cosine Decay learning rate scheduler.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from arabert.preprocess import ArabertPreprocessor
import numpy as np

# 1. Load Twitter-Specific AraBERT model and preprocessing
twitter_model_name = "aubmindlab/bert-base-arabertv02-twitter"
twitter_preprocessor = ArabertPreprocessor(twitter_model_name)
twitter_tokenizer = AutoTokenizer.from_pretrained(twitter_model_name)

# 2. Re-tokenize the dataset using the Twitter tokenizer
def preprocess_twitter_function(examples):
    texts = [twitter_preprocessor.preprocess(text) for text in examples["text"]]
    result = twitter_tokenizer(
        texts,
        truncation=True,
        max_length=128,
        padding="max_length",
    )
    if "label" in examples:
        result["label"] = [int(label) for label in examples["label"]]
    return result

twitter_tokenized = dataset.map(preprocess_twitter_function, batched=True)

# 3. Initialize Twitter-Specific Sequence Classifier
twitter_model = AutoModelForSequenceClassification.from_pretrained(
    twitter_model_name,
    num_labels=3
)

# 4. Define optimized training arguments (using a Cosine Scheduler)
advanced_training_args = TrainingArguments(
    output_dir="./twitter_model",
    num_train_epochs=5,                  # 5 epochs is usually enough and faster to converge
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,                  # Stable learning rate
    lr_scheduler_type="cosine",          # Cosine decay scheduler
    weight_decay=0.01,
    warmup_steps=100,                    # Use warmup_steps instead of warmup_ratio to avoid TypeError
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 5. Initialize Trainer with Early Stopping
advanced_trainer = Trainer(
    model=twitter_model,
    args=advanced_training_args,
    train_dataset=twitter_tokenized["train"],
    eval_dataset=twitter_tokenized["test"],
    compute_metrics=compute_metrics,
    processing_class=twitter_tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Run advanced training
advanced_trainer.train()

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/751k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=575, training_loss=0.478644440692404, metrics={'train_runtime': 350.4386, 'train_samples_per_second': 26.239, 'train_steps_per_second': 1.641, 'total_flos': 604831969002240.0, 'train_loss': 0.478644440692404, 'epoch': 5.0})

### 📈 Optimized Model Evaluation
We evaluate our best model (Twitter-AraBERT) to retrieve final test metrics.

In [ ]:
# Evaluate the final optimized model
advanced_results = advanced_trainer.evaluate()
print(f"Optimized Model Accuracy: {advanced_results['eval_accuracy']:.4f}")
print(f"Optimized Model F1 Macro: {advanced_results['eval_f1_macro']:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.527099,0.830953,5,0.709195,0.712280


Optimized Model Accuracy: 0.7092
Optimized Model F1 Macro: 0.7123


## 💾 Phase 4: Model Export & Evaluation Report Generation

We save our best-performing checkpoint and compile the overall training metrics into the workspace report.

In [ ]:
import os

# Save the optimized model and tokenizer
os.makedirs("models/arabert-twitter-optimized", exist_ok=True)
twitter_model.save_pretrained("models/arabert-twitter-optimized")
twitter_tokenizer.save_pretrained("models/arabert-twitter-optimized")

# Save final optimized metrics to the report
with open("reports/module-1.md", "a") as f:
    f.write("\n## Optimized Twitter-AraBERT Metrics\n\n")
    f.write(f"- **Accuracy:** {advanced_results['eval_accuracy']:.4f}\n")
    f.write(f"- **F1 Macro:** {advanced_results['eval_f1_macro']:.4f}\n")
    f.write(f"- **Model:** {twitter_model_name}\n")
    f.write(f"- **Epochs:** 5 (with Early Stopping)\n")
    f.write(f"- **Scheduler:** Cosine Decay\n")

print("Optimized model and report successfully saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Optimized model and report successfully saved!


### 📥 Download the Model to Your Local Device
Run the following cell to compress the model folder into a `.zip` archive and download it directly to your computer.

In [ ]:
import shutil
from google.colab import files

# Compress the saved model directory into a zip file
shutil.make_archive("arabert-twitter-optimized", "zip", "models/arabert-twitter-optimized")

# Trigger the download to your local device
files.download("arabert-twitter-optimized.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 💾 Export Model as Pickle (.pkl)
Run the following cell to serialize the fine-tuned PyTorch model using `pickle` and download it.

In [ ]:
import pickle
from google.colab import files

# Define the destination path for the pickle file
pickle_file_path = "models/arabert-twitter-optimized/model.pkl"

# Save the model object using pickle
with open(pickle_file_path, "wb") as f:
    pickle.dump(twitter_model, f)

print(f"Model successfully pickled and saved to {pickle_file_path}!")

# Trigger the download to your local device
files.download(pickle_file_path)

Model successfully pickled and saved to models/arabert-twitter-optimized/model.pkl!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 📦 Export Model to ONNX Format
We will install `optimum` and `onnx` to export our Hugging Face `transformers` model into the high-performance ONNX runtime format.

In [ ]:
!pip install optimum[onnxruntime] onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 12.4 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import shutil
from google.colab import files
import os

# Path settings
model_path = "models/arabert-twitter-optimized"
onnx_path = "models/arabert-twitter-onnx"
os.makedirs(onnx_path, exist_ok=True)

# Load the saved model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.eval()

# Create dummy input representative of our max_length (128)
dummy_input = tokenizer(
    "مرحبا بك",
    return_tensors="pt",
    padding="max_length",
    max_length=128,
    truncation=True
)

# Export using torch.onnx
output_onnx_file = os.path.join(onnx_path, "model.onnx")

torch.onnx.export(
    model,
    args=(dummy_input["input_ids"], dummy_input["attention_mask"]),
    f=output_onnx_file,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=12
)

# Save the tokenizer alongside the ONNX model
tokenizer.save_pretrained(onnx_path)
print(f"Model successfully exported directly to ONNX at: {output_onnx_file}")

# Compress and download the ONNX workspace
shutil.make_archive("arabert-twitter-onnx", "zip", onnx_path)
files.download("arabert-twitter-onnx.zip")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_652/2969336885.py:29: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1009 08:31:50.991000 652 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `BertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


...

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: sequence_length will not be used, since it shares the same shape constraints with another axis: sequence_length.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


Model successfully exported directly to ONNX at: models/arabert-twitter-onnx/model.onnx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>